# Lesson 2: 波形と音色の基礎

**コンパニオンノートブック** — 詳しい解説は本文 Lesson 2 を参照してください。

## セットアップ

In [ ]:
# --- 最初に1回だけ実行 ---
import sys
try:
    import google.colab
    !pip install -q japanize-matplotlib
    !git clone -q https://github.com/ggszk/simple-sound-programming.git
    sys.path.append('/content/simple-sound-programming')
except ImportError:
    sys.path.append('..')

from audio_lib.notebook import setup_environment
setup_environment()

## このレッスンで学ぶこと

- 基本的な波形（サイン波、矩形波、ノコギリ波、三角波）の違いを理解する
- 波形の違いが音色の違いになることを体験する
- Pythonで各波形を生成し、波形と音を比較する
- 倍音の考え方に触れる（次回以降への橋渡し）


In [ ]:
import numpy as np
from IPython.display import display
from audio_lib import sine_wave, square_wave, sawtooth_wave, triangle_wave, AudioSignal
from audio_lib.notebook import play_sound, plot_waveform, plot_spectrum

## 2.1 4つの基本波形

In [ ]:
freq = 440  # すべて同じ周波数（ラの音）
dur = 2.0   # 2秒間

sig_sin = sine_wave(freq, dur)
sig_sq  = square_wave(freq, dur)
sig_saw = sawtooth_wave(freq, dur)
sig_tri = triangle_wave(freq, dur)

display(play_sound(sig_sin, "サイン波"))
display(play_sound(sig_sq,  "矩形波"))
display(play_sound(sig_saw, "ノコギリ波"))
display(play_sound(sig_tri, "三角波"))

## 2.2 波形を見る

In [ ]:
plot_waveform(sig_sin, duration=0.01, title="サイン波 (440Hz)")
plot_waveform(sig_sq,  duration=0.01, title="矩形波 (440Hz)")
plot_waveform(sig_saw, duration=0.01, title="ノコギリ波 (440Hz)")
plot_waveform(sig_tri, duration=0.01, title="三角波 (440Hz)")

### サイン波（復習）

In [ ]:
sample_rate = 44100
t = np.linspace(0, dur, int(sample_rate * dur), endpoint=False)

data_sin = np.sin(2 * np.pi * freq * t)

### 矩形波

In [ ]:
# ft の小数部分（1周期の中での位置: 0.0〜1.0）
phase = (freq * t) % 1.0

# 前半は +1、後半は −1
data_sq = np.where(phase < 0.5, 1.0, -1.0)

### ノコギリ波

In [ ]:
data_saw = 2.0 * phase - 1.0

### 三角波

In [ ]:
data_tri = np.where(
    phase < 0.5,
    4.0 * phase - 1.0,    # 上昇部
    3.0 - 4.0 * phase,    # 下降部
)

### audio_lib との対応

In [ ]:
signal_manual = AudioSignal(data_sq, sample_rate)

## 2.4 波形を重ねて比較する

In [ ]:
import matplotlib.pyplot as plt

# 2周期分だけ表示（見やすさのため）
show_samples = int(sample_rate * 2 / freq)  # 2周期分のサンプル数
t_show = t[:show_samples] * 1000  # ミリ秒に変換

fig, axes = plt.subplots(4, 1, figsize=(12, 8), sharex=True)

waves = [
    (data_sin, "サイン波"),
    (data_sq,  "矩形波"),
    (data_saw, "ノコギリ波"),
    (data_tri, "三角波"),
]

for ax, (data, name) in zip(axes, waves):
    ax.plot(t_show, data[:show_samples])
    ax.set_ylabel(name)
    ax.set_ylim(-1.3, 1.3)
    ax.grid(True, alpha=0.3)

axes[-1].set_xlabel("時間 (ms)")
fig.suptitle("4つの基本波形（440Hz、2周期分）", fontsize=14)
plt.tight_layout()
plt.show()

## 2.5 周波数を変えて聞き比べる

In [ ]:
freq_low = 220

display(play_sound(sine_wave(freq_low, 2.0),     "サイン波 220Hz"))
display(play_sound(square_wave(freq_low, 2.0),    "矩形波 220Hz"))
display(play_sound(sawtooth_wave(freq_low, 2.0),  "ノコギリ波 220Hz"))
display(play_sound(triangle_wave(freq_low, 2.0),  "三角波 220Hz"))

In [ ]:
freq_high = 2000

display(play_sound(sine_wave(freq_high, 2.0),     "サイン波 2000Hz"))
display(play_sound(square_wave(freq_high, 2.0),    "矩形波 2000Hz"))
display(play_sound(sawtooth_wave(freq_high, 2.0),  "ノコギリ波 2000Hz"))
display(play_sound(triangle_wave(freq_high, 2.0),  "三角波 2000Hz"))

### 周波数スペクトラムで確認する

In [ ]:
plot_spectrum(sig_sin, max_freq=5000, title="サイン波のスペクトラム")
plot_spectrum(sig_sq,  max_freq=5000, title="矩形波のスペクトラム")
plot_spectrum(sig_saw, max_freq=5000, title="ノコギリ波のスペクトラム")
plot_spectrum(sig_tri, max_freq=5000, title="三角波のスペクトラム")

## 2.7 倍音の重ね合わせを体験する

In [ ]:
# サイン波を1つずつ足して矩形波に近づける
for n_harmonics in [1, 3, 5, 10]:
    data = np.zeros_like(t)
    for k in range(1, n_harmonics * 2, 2):  # 奇数次のみ: 1, 3, 5, ...
        data += (1.0 / k) * np.sin(2 * np.pi * k * freq * t)
    data *= 4.0 / np.pi

    signal = AudioSignal(data, sample_rate)
    display(play_sound(signal, f"奇数倍音 {n_harmonics}個"))

In [ ]:
# 波形の変化を可視化
fig, axes = plt.subplots(4, 1, figsize=(12, 8), sharex=True)
labels = [1, 3, 5, 10]

for ax, n_harmonics in zip(axes, labels):
    data = np.zeros_like(t)
    for k in range(1, n_harmonics * 2, 2):
        data += (1.0 / k) * np.sin(2 * np.pi * k * freq * t)
    data *= 4.0 / np.pi

    ax.plot(t_show, data[:show_samples])
    ax.set_ylabel(f"{n_harmonics}個")
    ax.set_ylim(-1.5, 1.5)
    ax.grid(True, alpha=0.3)

axes[-1].set_xlabel("時間 (ms)")
fig.suptitle("サイン波を重ねて矩形波に近づける", fontsize=14)
plt.tight_layout()
plt.show()

## 2.8 デューティ比 — 矩形波のバリエーション

In [ ]:
duty_cycles = [0.5, 0.25, 0.1]

for duty in duty_cycles:
    sig = square_wave(freq, 2.0, duty_cycle=duty)
    display(play_sound(sig, f"矩形波 デューティ比={duty}"))

In [ ]:
fig, axes = plt.subplots(3, 1, figsize=(12, 6), sharex=True)

for ax, duty in zip(axes, duty_cycles):
    sig = square_wave(freq, dur, duty_cycle=duty)
    show_data = sig.data[:show_samples]
    ax.plot(t_show, show_data)
    ax.set_ylabel(f"duty={duty}")
    ax.set_ylim(-1.3, 1.3)
    ax.grid(True, alpha=0.3)

axes[-1].set_xlabel("時間 (ms)")
fig.suptitle("デューティ比による矩形波の変化", fontsize=14)
plt.tight_layout()
plt.show()